# Cogni AP Tutor — base vs. tuned evaluation

Run this notebook top-to-bottom in **Google Colab with a GPU runtime**. It generates predictions from the untouched base model and the trained model on all 756 test examples, scores the strict seven-section behavior, and saves a comparison in Google Drive. Generation files are resumable.


## 1. Confirm the GPU and install dependencies


In [ ]:
!nvidia-smi
!pip -q install -U transformers accelerate bitsandbytes sentencepiece


## 2. Mount Drive and verify the inputs


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, re, time, gc
from pathlib import Path

TEST_FILE = Path('/content/drive/MyDrive/cogni/production_v1/test.jsonl')
TUNED_MODEL = Path('/content/drive/MyDrive/cogni/models/ap_tutor_qwen3_1_7b_v1/merged_16bit')
RESULTS_DIR = Path('/content/drive/MyDrive/cogni/evaluation/ap_tutor_base_vs_tuned_v1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert TEST_FILE.exists(), TEST_FILE
assert TUNED_MODEL.exists(), TUNED_MODEL
test_rows = [json.loads(line) for line in TEST_FILE.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(test_rows) == 756, f'Expected 756 untouched test rows, found {len(test_rows)}'
test_ids = [str(r.get('example_id') or r.get('id') or '') for r in test_rows]
assert all(test_ids), 'One or more test rows has no id/example_id'
assert len(set(test_ids)) == len(test_rows), 'Duplicate test IDs'
print('Test examples:', len(test_rows))
print('Tuned model:', TUNED_MODEL)
print('Results:', RESULTS_DIR)


## 3. Define the prompt and resumable prediction function


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then rerun.'

SYSTEM_PROMPT = (
    'You are an AP English Language logical-fallacy tutor. Follow the required '
    'seven-section instructional sequence exactly: Argument Summary, Assumptions, '
    'Primary Fallacy, Why This Applies, Cross-Domain Analogy, Transfer Check, and '
    'Reflective Question.'
)

def example_id(row, index):
    return str(row.get('example_id') or row.get('id') or f'test-{index:06d}')

def input_messages(row):
    messages = row.get('messages')
    if isinstance(messages, list):
        usable = [m for m in messages if isinstance(m, dict) and m.get('role') != 'assistant']
        if usable:
            return usable
    argument = str(row.get('argument_text') or row.get('essay') or row.get('prompt') or '').strip()
    return [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': argument}]

def read_completed(path):
    completed = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                row = json.loads(line)
                completed[str(row['example_id'])] = row
    return completed

def generate_predictions(model_source, model_name, output_file, batch_size=8, max_new_tokens=900):
    output_file = Path(output_file)
    completed = read_completed(output_file)
    pending = [(i, row) for i, row in enumerate(test_rows, 1) if example_id(row, i) not in completed]
    print(f'{model_name}: {len(completed)} already saved; {len(pending)} pending')
    if not pending:
        return

    tokenizer = AutoTokenizer.from_pretrained(str(model_source), trust_remote_code=True)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        str(model_source), trust_remote_code=True, device_map='auto', torch_dtype=torch.bfloat16
    )
    model.eval()
    started = time.time()

    with output_file.open('a', encoding='utf-8') as handle:
        for start in range(0, len(pending), batch_size):
            batch = pending[start:start + batch_size]
            prompts = []
            for _, row in batch:
                messages = input_messages(row)
                try:
                    prompt = tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
                    )
                except TypeError:
                    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                prompts.append(prompt)

            encoded = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=2048)
            encoded = {k: v.to(model.device) for k, v in encoded.items()}
            with torch.inference_mode():
                generated = model.generate(
                    **encoded, max_new_tokens=max_new_tokens, do_sample=False,
                    pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
                )
            input_width = encoded['input_ids'].shape[1]
            responses = tokenizer.batch_decode(generated[:, input_width:], skip_special_tokens=True)

            for (index, row), response in zip(batch, responses):
                record = {
                    'run_id': 'ap-test-base-vs-tuned-v1', 'model_id': model_name,
                    'example_id': example_id(row, index), 'response': response.strip()
                }
                handle.write(json.dumps(record, ensure_ascii=False) + '\n')
            handle.flush()

            done = min(start + len(batch), len(pending))
            elapsed = time.time() - started
            rate = done / elapsed if elapsed else 0
            eta_min = (len(pending) - done) / rate / 60 if rate else 0
            if done % 40 == 0 or done == len(pending):
                print(f'{model_name}: {done}/{len(pending)} new | ETA {eta_min:.1f} min')

    del model, tokenizer, encoded, generated
    gc.collect()
    torch.cuda.empty_cache()
    print(f'{model_name} complete: {output_file}')


## 4. Generate base-model predictions
This may take a while. If Colab disconnects, rerun the notebook: completed rows are skipped.


In [ ]:
BASE_OUTPUT = RESULTS_DIR / 'base_predictions.jsonl'
generate_predictions('Qwen/Qwen3-1.7B', 'Qwen3-1.7B-base', BASE_OUTPUT)


## 5. Generate tuned-model predictions


In [ ]:
TUNED_OUTPUT = RESULTS_DIR / 'tuned_predictions.jsonl'
generate_predictions(TUNED_MODEL, 'ap-tutor-qwen3-1.7b-v1', TUNED_OUTPUT)


## 6. Score the strict seven-section behavior
The strict pass requires all seven headings in the correct order plus automated checks for one primary fallacy, one analogy, an unanswered transfer question, and exactly one final reflective question. This deterministic score is reproducible; nuanced quality should later be checked on a sample by a human or judge model.


In [ ]:
HEADINGS = [
    'Argument Summary', 'Assumptions', 'Primary Fallacy', 'Why This Applies',
    'Cross-Domain Analogy', 'Transfer Check', 'Reflective Question'
]

def section_text(text, heading, next_heading=None):
    start_match = re.search(rf'(?im)^\s*(?:#{{1,6}}\s*)?(?:\d+[.)]\s*)?{re.escape(heading)}\s*:?[ \t]*$', text)
    if not start_match:
        return ''
    start = start_match.end()
    if next_heading:
        end_match = re.search(rf'(?im)^\s*(?:#{{1,6}}\s*)?(?:\d+[.)]\s*)?{re.escape(next_heading)}\s*:?[ \t]*$', text[start:])
        end = start + end_match.start() if end_match else len(text)
    else:
        end = len(text)
    return text[start:end].strip()

def score_response(response):
    text = response.split('</think>', 1)[-1].strip()
    positions = []
    sections = {}
    for i, heading in enumerate(HEADINGS):
        match = re.search(rf'(?im)^\s*(?:#{{1,6}}\s*)?(?:\d+[.)]\s*)?{re.escape(heading)}\s*:?[ \t]*$', text)
        positions.append(match.start() if match else -1)
        sections[heading] = section_text(text, heading, HEADINGS[i + 1] if i + 1 < len(HEADINGS) else None)

    heading_present = [p >= 0 for p in positions]
    correct_order = all(positions[i] < positions[i + 1] for i in range(6)) if all(heading_present) else False
    nonempty = [bool(sections[h].strip()) for h in HEADINGS]
    primary = sections['Primary Fallacy']
    analogy = sections['Cross-Domain Analogy']
    transfer = sections['Transfer Check']
    reflective = sections['Reflective Question']

    one_primary = bool(primary) and not bool(re.search(r'(?i)\b(?:secondary|also|another|alternatively)\b', primary))
    one_analogy = bool(analogy) and not bool(re.search(r'(?i)\b(?:second|another) analogy\b', analogy))
    transfer_question = transfer.count('?') >= 1 and not bool(re.search(r'(?i)\b(?:answer|classification)\s*(?:is|:)\b', transfer))
    final_reflection = reflective.count('?') == 1 and text.rstrip().endswith('?')

    checks = {
        **{f'S{i+1}_{HEADINGS[i].lower().replace(" ", "_")}': heading_present[i] and nonempty[i] for i in range(7)},
        'correct_section_order': correct_order, 'exactly_one_primary_fallacy': one_primary,
        'exactly_one_analogy': one_analogy, 'transfer_without_answer': transfer_question,
        'exactly_one_final_reflective_question': final_reflection,
    }
    return checks, all(checks.values())

def score_file(prediction_file, model_name):
    predictions = read_completed(prediction_file)
    assert len(predictions) == len(test_rows), f'{model_name}: expected 756 predictions, found {len(predictions)}'
    details, totals = [], {}
    for eid, pred in predictions.items():
        checks, passed = score_response(str(pred.get('response', '')))
        details.append({'example_id': eid, 'passed': passed, 'checks': checks})
        for key, value in checks.items():
            totals[key] = totals.get(key, 0) + int(value)
    n = len(details)
    summary = {
        'model': model_name, 'example_count': n,
        'strict_pass_rate': sum(d['passed'] for d in details) / n,
        'per_check_rate': {key: value / n for key, value in totals.items()},
    }
    return summary, details

base_summary, base_details = score_file(BASE_OUTPUT, 'Qwen3-1.7B-base')
tuned_summary, tuned_details = score_file(TUNED_OUTPUT, 'ap-tutor-qwen3-1.7b-v1')
print('Scoring complete.')


## 7. Compare and save the results


In [ ]:
comparison = {
    'test_file': str(TEST_FILE), 'test_examples': len(test_rows),
    'base': base_summary, 'tuned': tuned_summary,
    'strict_pass_rate_delta': tuned_summary['strict_pass_rate'] - base_summary['strict_pass_rate'],
    'per_check_delta': {
        key: tuned_summary['per_check_rate'][key] - base_summary['per_check_rate'][key]
        for key in base_summary['per_check_rate']
    },
}

(RESULTS_DIR / 'base_scores.json').write_text(json.dumps(base_details, indent=2), encoding='utf-8')
(RESULTS_DIR / 'tuned_scores.json').write_text(json.dumps(tuned_details, indent=2), encoding='utf-8')
(RESULTS_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')

print(f"Base strict pass:  {base_summary['strict_pass_rate']:.1%}")
print(f"Tuned strict pass: {tuned_summary['strict_pass_rate']:.1%}")
print(f"Improvement:       {comparison['strict_pass_rate_delta']:+.1%}")
print('\nPer-check comparison:')
for key in base_summary['per_check_rate']:
    b = base_summary['per_check_rate'][key]
    t = tuned_summary['per_check_rate'][key]
    print(f'{key:42s} base={b:6.1%}  tuned={t:6.1%}  delta={t-b:+6.1%}')
print('\nSaved:', RESULTS_DIR / 'comparison.json')


## Finished
Download `comparison.json`, or paste the final cell output into Codex for interpretation and failure analysis. Keep the test set out of any future training data.
